In [583]:
# import libraries
import pandas as pd
from pathlib import Path
import glob 

## 1. Import file

In [584]:
#load dataset
raw_path = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/Raw")

In [585]:
orders = pd.read_csv(raw_path / "olist_orders_dataset.csv")
customers = pd.read_csv(raw_path / "olist_customers_dataset.csv")
order_items = pd.read_csv(raw_path / "olist_order_items_dataset.csv")
products = pd.read_csv(raw_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_path / "olist_sellers_dataset.csv")
payments = pd.read_csv(raw_path / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_path / "olist_order_reviews_dataset.csv")
geolocation = pd.read_csv(raw_path / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(raw_path / "product_category_name_translation.csv")

In [586]:
print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("geolocation:", geolocation.shape)
print("category_translation:", category_translation.shape)

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
products: (32951, 9)
sellers: (3095, 4)
payments: (103886, 5)
reviews: (99224, 7)
geolocation: (1000163, 5)
category_translation: (71, 2)


## 2. Datatype Conversion

In [587]:
orders.dtypes

order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

In [588]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])
    

In [589]:
orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [590]:
order_items.dtypes

order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

In [591]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [592]:
order_items.dtypes

order_id                       object
order_item_id                   int64
product_id                     object
seller_id                      object
shipping_limit_date    datetime64[ns]
price                         float64
freight_value                 float64
dtype: object

In [593]:
products.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [594]:
sellers.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [595]:
payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [596]:
customers.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

## 3. Datatype Quality Cleaning

In [597]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [598]:
orders["order_id"].is_unique

True

In [599]:
(orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum()

np.int64(0)

In [600]:
(orders["order_approved_at"] < orders["order_purchase_timestamp"]).sum()

np.int64(0)

In [601]:
(
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
).sum()

np.int64(23)

In [602]:
invalid_delivery = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
]

invalid_delivery[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


```Data quality rule:
If delivered_customer_date < delivered_carrier_date,
the order is considered temporally inconsistent and excluded. 

In [603]:
invalid_delivery = (
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
)

orders = orders.loc[~invalid_delivery].copy()

In [604]:
(
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
).sum()

np.int64(0)

In [605]:
orders.shape

(99418, 8)

In [606]:
order_items.isna().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [607]:
(order_items['price']<0).sum()

np.int64(0)

In [608]:
(order_items['freight_value']<0).sum()

np.int64(0)

In [609]:
item_count =(
    order_items.groupby("order_id")["order_item_id"].count()
)

In [610]:
item_count.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: order_item_id, dtype: int64

In [611]:
item_features = item_count.to_frame("item_count")

In [612]:
item_features.head()

,item_count
order_id,
00010242fe8c5a6d1ba2dd792cb16214,1
00018f77f2f0320c557190d7a144bdd3,1
000229ec398224ef6ca0657da4fc703e,1
00024acbcdf0a6daa1e931b038114c75,1
00042b26cf59d7ce69dfabb4e55b4fd9,1


In [613]:
item_value_features = (
    order_items
    .groupby("order_id")[["price", "freight_value"]]
    .sum()
)

item_value_features.head()

,price,freight_value
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,239.90,19.93
000229ec398224ef6ca0657da4fc703e,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14


In [614]:
item_value_features = item_value_features.rename(columns={"price": "total_price", "freight_value": "total_freight_value"})

In [615]:
item_features = item_features.join(item_value_features)

In [616]:
item_features.head()

,item_count,total_price,total_freight_value
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [617]:
n_unique_products = order_items.groupby('order_id')['product_id'].nunique()

In [618]:
n_unique_products.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: product_id, dtype: int64

In [619]:
n_unique_sellers = order_items.groupby('order_id')['seller_id'].nunique()

In [620]:
n_unique_sellers.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: seller_id, dtype: int64

In [621]:
item_features = pd.concat([
    item_features,
    item_value_features,
    n_unique_products.rename("n_unique_products"),
    n_unique_sellers.rename("n_unique_sellers")
],
axis=1
)

In [622]:
item_features.head()

,item_count,total_price,total_freight_value,total_price,total_freight_value,n_unique_products,n_unique_sellers
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90,13.29,1,1
00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90,19.93,1,1
000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00,17.87,1,1
00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99,12.79,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90,18.14,1,1


In [623]:
item_features.shape

(98666, 7)

In [624]:
item_features = item_features.loc[:, ~item_features.columns.duplicated()]

In [625]:
item_features.head()

,item_count,total_price,total_freight_value,n_unique_products,n_unique_sellers
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1,1
00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1,1
000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1,1
00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1,1


In [626]:
item_features.shape

(98666, 5)

In [627]:
order_item_products = order_items[
    ["order_id", "product_id"]
].merge(
    products,
    on="product_id",
    how="left"
)

In [628]:
order_item_products.head()

,order_id,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0


In [629]:
product_features = order_item_products[
    [
        "order_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

In [630]:
product_features.head()

,order_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,moveis_decoracao,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,perfumaria,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,ferramentas_jardim,3750.0,35.0,40.0,30.0


In [631]:
product_weight = product_features.groupby('order_id')['product_weight_g'].sum().rename('total_product_weight_g')

product_weight.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214      650.0
00018f77f2f0320c557190d7a144bdd3    30000.0
000229ec398224ef6ca0657da4fc703e     3050.0
00024acbcdf0a6daa1e931b038114c75      200.0
00042b26cf59d7ce69dfabb4e55b4fd9     3750.0
Name: total_product_weight_g, dtype: float64

In [632]:
product_length = product_features.groupby('order_id')['product_length_cm'].mean().rename('average_product_length_cm')

product_length.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    28.0
00018f77f2f0320c557190d7a144bdd3    50.0
000229ec398224ef6ca0657da4fc703e    33.0
00024acbcdf0a6daa1e931b038114c75    16.0
00042b26cf59d7ce69dfabb4e55b4fd9    35.0
Name: average_product_length_cm, dtype: float64

In [633]:
product_height = product_features.groupby('order_id')['product_height_cm'].mean().rename('average_product_height_cm')
product_height.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214     9.0
00018f77f2f0320c557190d7a144bdd3    30.0
000229ec398224ef6ca0657da4fc703e    13.0
00024acbcdf0a6daa1e931b038114c75    10.0
00042b26cf59d7ce69dfabb4e55b4fd9    40.0
Name: average_product_height_cm, dtype: float64

In [634]:
product_width = product_features.groupby('order_id')['product_width_cm'].mean().rename('average_product_width_cm')
product_width.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    14.0
00018f77f2f0320c557190d7a144bdd3    40.0
000229ec398224ef6ca0657da4fc703e    33.0
00024acbcdf0a6daa1e931b038114c75    15.0
00042b26cf59d7ce69dfabb4e55b4fd9    30.0
Name: average_product_width_cm, dtype: float64

In [635]:
n_unique_categories = product_features.groupby('order_id')['product_category_name'].nunique().rename('n_unique_categories')
n_unique_categories.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: n_unique_categories, dtype: int64

In [636]:
product_features_agg = pd.concat([
    product_weight,
    product_length,
    product_height,
    product_width,
    n_unique_categories
],
axis =1
)

In [637]:
product_features_agg.head()

,total_product_weight_g,average_product_length_cm,average_product_height_cm,average_product_width_cm,n_unique_categories
order_id,,,,,
00010242fe8c5a6d1ba2dd792cb16214,650.0,28.0,9.0,14.0,1
00018f77f2f0320c557190d7a144bdd3,30000.0,50.0,30.0,40.0,1
000229ec398224ef6ca0657da4fc703e,3050.0,33.0,13.0,33.0,1
00024acbcdf0a6daa1e931b038114c75,200.0,16.0,10.0,15.0,1
00042b26cf59d7ce69dfabb4e55b4fd9,3750.0,35.0,40.0,30.0,1


In [638]:
product_features_agg.shape

(98666, 5)

In [639]:
sellers.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

In [640]:
order_item_sellers = order_items[
    ["order_id", "seller_id"]
].merge(
    sellers,
    on="seller_id",
    how="left"
)

In [641]:
order_item_sellers.head()

,order_id,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR


In [642]:
n_unique_seller_city = order_item_sellers.groupby('order_id')['seller_city'].nunique().rename('n_unique_seller_city')
n_unique_seller_city.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: n_unique_seller_city, dtype: int64

In [643]:
n_unique_seller_state = order_item_sellers.groupby('order_id')['seller_state'].nunique().rename('n_unique_seller_state')
n_unique_seller_state.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: n_unique_seller_state, dtype: int64

In [644]:
order_item_sellers.columns.tolist()

['order_id',
 'seller_id',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state']

In [645]:
seller_features_agg = pd.concat([
    n_unique_seller_city,
    n_unique_seller_state,
], axis=1)

In [646]:
seller_features_agg.head()

,n_unique_seller_city,n_unique_seller_state
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,1,1
00018f77f2f0320c557190d7a144bdd3,1,1
000229ec398224ef6ca0657da4fc703e,1,1
00024acbcdf0a6daa1e931b038114c75,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,1


In [647]:
seller_features_agg.shape

(98666, 2)

In [648]:
payments.dtypes

order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

In [649]:
total_payment_value = (
    payments
    .groupby("order_id")["payment_value"]
    .sum()
    .rename("total_payment_value")
)

total_payment_value.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214     72.19
00018f77f2f0320c557190d7a144bdd3    259.83
000229ec398224ef6ca0657da4fc703e    216.87
00024acbcdf0a6daa1e931b038114c75     25.78
00042b26cf59d7ce69dfabb4e55b4fd9    218.04
Name: total_payment_value, dtype: float64

In [650]:
payment_installments =(
    payments
    .groupby("order_id")["payment_installments"]
    .sum().rename("max_payment_installments")
)

payment_installments.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    2
00018f77f2f0320c557190d7a144bdd3    3
000229ec398224ef6ca0657da4fc703e    5
00024acbcdf0a6daa1e931b038114c75    2
00042b26cf59d7ce69dfabb4e55b4fd9    3
Name: max_payment_installments, dtype: int64

In [651]:
n_unique_pyament_types=(
    payments
    .groupby('order_id')['payment_installments'].nunique().rename('n_unique_payment_types')
)

n_unique_pyament_types.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: n_unique_payment_types, dtype: int64

In [652]:
payment_features_agg = pd.concat([
    total_payment_value,
    payment_installments,
    n_unique_pyament_types
],axis=1)

In [653]:
payment_features_agg.head()

,total_payment_value,max_payment_installments,n_unique_payment_types
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
00018f77f2f0320c557190d7a144bdd3,259.83,3,1
000229ec398224ef6ca0657da4fc703e,216.87,5,1
00024acbcdf0a6daa1e931b038114c75,25.78,2,1
00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


In [654]:
payment_features_agg.shape

(99440, 3)

In [655]:
customers.dtypes

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

In [656]:
customer_features = customers[
    ["customer_id", "customer_city", "customer_state"]
]

customer_features.head()

,customer_id,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,campinas,SP


In [657]:
customer_features.shape

(99441, 3)

In [658]:
customer_features["customer_id"].is_unique

True

In [659]:
final_df = orders.merge(
    item_features,
    on="order_id",
    how="left"
)

print(final_df.shape)
print(final_df["order_id"].is_unique)

(99418, 13)
True


In [660]:
final_df = final_df.merge(
    product_features_agg,
    on="order_id",
    how="left"
)

print(final_df.shape)
print(final_df["order_id"].is_unique)

(99418, 18)
True


In [661]:
final_df = final_df.merge(
    seller_features_agg,
    on='order_id',
    how='left'
)
print(final_df.shape)
print(final_df["order_id"].is_unique)

(99418, 20)
True


In [662]:
seller_features_agg.head()

,n_unique_seller_city,n_unique_seller_state
order_id,,
00010242fe8c5a6d1ba2dd792cb16214,1,1
00018f77f2f0320c557190d7a144bdd3,1,1
000229ec398224ef6ca0657da4fc703e,1,1
00024acbcdf0a6daa1e931b038114c75,1,1
00042b26cf59d7ce69dfabb4e55b4fd9,1,1


In [663]:
final_df = final_df.merge(
    payment_features_agg,
    on="order_id",
    how="left"
)

print(final_df.shape)
print(final_df["order_id"].is_unique)

(99418, 23)
True


In [664]:
final_df = final_df.merge(
    customer_features,
    on="customer_id",
    how="left"
)

print(final_df.shape)
print(final_df["order_id"].is_unique)

(99418, 25)
True


## 4. Construct Target

In [665]:
final_df['delivery_delay']=(
    final_df['order_delivered_customer_date']
    - final_df['order_estimated_delivery_date']
)

In [666]:
final_df['delivery_delay'].describe()

count                           96453
mean     -12 days +19:46:23.498087152
std        10 days 04:24:33.999809828
min               -147 days +23:36:47
25%                -17 days +18:08:54
50%                -12 days +01:19:29
75%                 -7 days +14:40:55
max                 188 days 23:24:07
Name: delivery_delay, dtype: object

In [667]:
final_df['delivery_delay_days'] = (
    final_df['delivery_delay'].dt.total_seconds() / (24 * 60 * 60)
)

In [668]:
final_df['delivery_delay_days'].describe()

count    96453.000000
mean       -11.176117
std         10.183727
min       -146.016123
25%        -16.243819
50%        -11.944803
75%         -6.388252
max        188.975081
Name: delivery_delay_days, dtype: float64

In [669]:
final_df["is_delayed"] = (
    final_df["delivery_delay_days"]
    .gt(0)
    .astype("boolean")
)

In [670]:
final_df.loc[
    final_df['delivery_delay_days'].isna(),
    'is_delayed'
] = pd.NA

In [671]:
final_df["is_delayed"].value_counts(dropna=False)

is_delayed
False    88626
True      7827
<NA>      2965
Name: count, dtype: Int64

In [672]:
final_df["order_id"].is_unique

True

In [673]:
final_df.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'item_count',
 'total_price',
 'total_freight_value',
 'n_unique_products',
 'n_unique_sellers',
 'total_product_weight_g',
 'average_product_length_cm',
 'average_product_height_cm',
 'average_product_width_cm',
 'n_unique_categories',
 'n_unique_seller_city',
 'n_unique_seller_state',
 'total_payment_value',
 'max_payment_installments',
 'n_unique_payment_types',
 'customer_city',
 'customer_state',
 'delivery_delay',
 'delivery_delay_days',
 'is_delayed']

In [674]:
final_df.isna().sum().sort_values(ascending=False)

is_delayed                       2965
delivery_delay_days              2965
delivery_delay                   2965
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
average_product_height_cm         791
average_product_width_cm          791
average_product_length_cm         791
item_count                        775
total_product_weight_g            775
n_unique_seller_state             775
total_price                       775
n_unique_categories               775
n_unique_seller_city              775
n_unique_sellers                  775
n_unique_products                 775
total_freight_value               775
order_approved_at                 160
total_payment_value                 1
max_payment_installments            1
n_unique_payment_types              1
customer_id                         0
order_estimated_delivery_date       0
customer_city                       0
customer_state                      0
order_purchase_timestamp            0
order_status

In [675]:
final_df.shape

(99418, 28)

In [676]:
final_df.dtypes

order_id                                  object
customer_id                               object
order_status                              object
order_purchase_timestamp          datetime64[ns]
order_approved_at                 datetime64[ns]
order_delivered_carrier_date      datetime64[ns]
order_delivered_customer_date     datetime64[ns]
order_estimated_delivery_date     datetime64[ns]
item_count                               float64
total_price                              float64
total_freight_value                      float64
n_unique_products                        float64
n_unique_sellers                         float64
total_product_weight_g                   float64
average_product_length_cm                float64
average_product_height_cm                float64
average_product_width_cm                 float64
n_unique_categories                      float64
n_unique_seller_city                     float64
n_unique_seller_state                    float64
total_payment_value 

# Save final file 

In [ ]:
'''processed_path = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset")
processed_path.mkdir(parents=True, exist_ok=True)

final_df.to_parquet(
    processed_path / "olist_clean.parquet",
    index=False
)'''

In [678]:
path = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/olist_clean.parquet")


In [679]:
pd.read_parquet(path).head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,item_count,total_price,...,n_unique_seller_city,n_unique_seller_state,total_payment_value,max_payment_installments,n_unique_payment_types,customer_city,customer_state,delivery_delay,delivery_delay_days,is_delayed
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,...,1.0,1.0,38.71,3.0,1.0,sao paulo,SP,-8 days +21:25:13,-7.107488,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,...,1.0,1.0,141.46,1.0,1.0,barreiras,BA,-6 days +15:27:45,-5.355729,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,...,1.0,1.0,179.12,3.0,1.0,vianopolis,GO,-18 days +18:06:29,-17.245498,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,...,1.0,1.0,72.20,1.0,1.0,sao goncalo do amarante,RN,-13 days +00:28:42,-12.980069,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,...,1.0,1.0,28.62,1.0,1.0,santo andre,SP,-10 days +18:17:02,-9.238171,False


In [680]:
pd.read_parquet(path).shape

(99418, 28)

In [681]:
product_features = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_photos_qty"
]

products[product_features].isna().sum()

product_weight_g        2
product_length_cm       2
product_height_cm       2
product_width_cm        2
product_photos_qty    610
dtype: int64

In [682]:
products[product_features].describe().T

,count,mean,std,min,25%,50%,75%,max
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0


In [683]:
product_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products[product_cols].isna().sum()

product_weight_g     2
product_length_cm    2
product_height_cm    2
product_width_cm     2
dtype: int64

In [684]:
missing_products = products[
    products[product_cols].isna().any(axis=1)
]["product_id"]

missing_order_items = order_items[
    order_items["product_id"].isin(missing_products)
]

missing_order_items.shape

(18, 7)

In [685]:
missing_order_items["order_id"].nunique()

16

# Add distance feature

In [686]:
order_item_sellers.head()

,order_id,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR


In [687]:
seller_geo = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        seller_lat=("geolocation_lat", "mean"),
        seller_lng=("geolocation_lng", "mean")
    )
)

In [688]:
customer_geo = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        customer_lat=("geolocation_lat", "mean"),
        customer_lng=("geolocation_lng", "mean")
    )
)

In [689]:
order_item_sellers = order_item_sellers.merge(
    seller_geo,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
).drop(
    columns="geolocation_zip_code_prefix"
)

In [690]:
order_item_sellers.head()

,order_id,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_lat,seller_lng
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,27277,volta redonda,SP,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,3471,sao paulo,SP,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,37564,borda da mata,MG,-22.262584,-46.171124
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,14403,franca,SP,-20.553624,-47.387359
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,87900,loanda,PR,-22.929384,-53.135873


In [691]:
order_item_sellers[["seller_lat", "seller_lng"]].isna().sum()

seller_lat    253
seller_lng    253
dtype: int64

In [692]:
order_item_sellers.shape

(112650, 7)

In [693]:
geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [694]:
customer_geo.head()

,geolocation_zip_code_prefix,customer_lat,customer_lng
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


In [695]:
customer_geo.isna().sum()

geolocation_zip_code_prefix    0
customer_lat                   0
customer_lng                   0
dtype: int64

In [696]:
order_customer = (
    orders[
        ["order_id", "customer_id"]
    ]
    .merge(
        customers[
            ["customer_id", "customer_zip_code_prefix"]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        customer_geo,
        left_on="customer_zip_code_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

In [697]:
order_customer.head()

,order_id,customer_id,customer_zip_code_prefix,geolocation_zip_code_prefix,customer_lat,customer_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3149,3149.0,-23.576983,-46.587161
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,47813,47813.0,-12.177924,-44.660711
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,75265,75265.0,-16.745150,-48.514783
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,59296,59296.0,-5.774190,-35.271143
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,9195,9195.0,-23.676370,-46.514627


In [698]:
order_customer[["customer_lat", "customer_lng"]].isna().sum()

customer_lat    278
customer_lng    278
dtype: int64

In [699]:
order_customer.shape

(99418, 6)

In [700]:
orders[
    ["order_id", "customer_id"]
].shape

(99418, 2)

In [701]:
order_customer["order_id"].nunique()

99418

In [702]:
customers["customer_zip_code_prefix"].isna().sum()

np.int64(0)

In [703]:
customers["customer_zip_code_prefix"].nunique()

14994

In [704]:
order_distance = order_customer.merge(
    order_item_sellers[
        [
            "order_id",
            "seller_id",
            "seller_zip_code_prefix",
            "seller_lat",
            "seller_lng"
        ]
    ],
    on="order_id",
    how="left"
)

In [705]:
order_distance.head()

,order_id,customer_id,customer_zip_code_prefix,geolocation_zip_code_prefix,customer_lat,customer_lng,seller_id,seller_zip_code_prefix,seller_lat,seller_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3149,3149.0,-23.576983,-46.587161,3504c0cb71d7fa48d967e0e4c94d59d9,9350.0,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,47813,47813.0,-12.177924,-44.660711,289cdb325fb7e7f891c38608bf9e0962,31570.0,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,75265,75265.0,-16.745150,-48.514783,4869f7a5dfa277a7dca6462dcf3b52b2,14840.0,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,59296,59296.0,-5.774190,-35.271143,66922902710d126a0e7d26b0e3805106,31842.0,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,9195,9195.0,-23.676370,-46.514627,2c9e548be18521d1c43cde1c582c6de8,8752.0,-23.543395,-46.262086


In [706]:
order_distance.shape

(113375, 10)

In [707]:
order_distance[
    ["customer_lat", "customer_lng", "seller_lat", "seller_lng"]
].isna().sum()

customer_lat     306
customer_lng     306
seller_lat      1028
seller_lng      1028
dtype: int64

In [708]:
order_customer["order_id"].duplicated().sum()

np.int64(0)

In [709]:
order_customer[
    order_customer["order_id"].duplicated(keep=False)
].sort_values("order_id").head(20)

,order_id,customer_id,customer_zip_code_prefix,geolocation_zip_code_prefix,customer_lat,customer_lng


In [710]:
order_customer["order_id"].value_counts().head(10)

order_id
e481f51cbdc54678b7cc49136f2d6af7    1
f01059d0d674e1282df4e8fbbe015aa2    1
fbc17f0f2a2125054d5ac5c22d2d5120    1
9373150545066777b1cd2bc20e93cf8e    1
917399e96f92268dfa2c0351b1b75fba    1
ed1691ef26bd8279bd5946561af1ff0d    1
dc3006aa87f57332aaff74c57a5e094d    1
f53eae1ce47dc68e8da117dd0d7feef1    1
94bce2ab6f38b41d29ebbd9d755677bf    1
632f22d24375715fbfa8c0ae2e5d35b7    1
Name: count, dtype: int64

In [711]:
order_items[["order_id", "seller_id"]]

,order_id,seller_id
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87
...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,b8bc237ba3788b23da09c0f1f3a3288c
112646,fffcd46ef2263f404302a634eb57f7eb,f3c38ab652836d21de61fb8314b69182
112647,fffce4705a9662cd70adb13d4a31832d,c3cfdc648177fdbbbb35635a37472c53
112648,fffe18544ffabc95dfada21779c9644f,2b3e4a2a3ea8e01938cabda2a3e5cc79


In [712]:
order_item_sellers.shape

(112650, 7)

In [713]:
order_items[["order_id", "seller_id"]].shape

(112650, 2)

In [714]:
sellers["seller_id"].duplicated().sum()

np.int64(0)

In [715]:
seller_geo["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(0)

In [716]:
order_distance = order_customer.merge(
    order_item_sellers[
        [
            "order_id",
            "seller_id",
            "seller_zip_code_prefix",
            "seller_lat",
            "seller_lng"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_many"
)

In [717]:
order_item_sellers.groupby("order_id").size().sum()

np.int64(112650)

In [718]:
order_distance.groupby("order_id").size().sum()

np.int64(113375)

In [719]:
check = (
    order_item_sellers.groupby("order_id").size()
    .rename("before")
    .to_frame()
    .join(
        order_distance.groupby("order_id").size().rename("after"),
        how="outer"
    )
)

check[check["before"] != check["after"]].head(20)

,before,after
order_id,,
0010dedd556712d7bb69a19cb7bbd37a,NaN,1.0
00a500bc03bc4ec968e574c2553bed4b,NaN,1.0
00b1cb0320190ca0daa2c88b35206009,NaN,1.0
00bca4adac549020c1273714d04d0208,NaN,1.0
00d0ffd14774da775ac832ba8520510f,NaN,1.0
00daac8efd71674d62356c2a306d1e4c,NaN,1.0
0130f0f71fb0e831d18e6a3b33a3a50c,NaN,1.0
016706d732e79b49035275c048edb2c4,NaN,1.0
01cb6d702e5233235f4125309d184bf4,NaN,1.0


In [720]:
order_distance = order_item_sellers[
    [
        "order_id",
        "seller_id",
        "seller_zip_code_prefix",
        "seller_lat",
        "seller_lng"
    ]
].merge(
    order_customer[
        [
            "order_id",
            "customer_lat",
            "customer_lng"
        ]
    ],
    on="order_id",
    how="left"
)

In [721]:
order_distance.shape

(112650, 7)

In [722]:
order_distance[
    ["customer_lat", "customer_lng", "seller_lat", "seller_lng"]
].isna().sum()

customer_lat    352
customer_lng    352
seller_lat      253
seller_lng      253
dtype: int64

In [723]:
from numpy import radians, sin, cos, sqrt, arctan2

valid_distance = order_distance.dropna(
    subset=[
        "customer_lat",
        "customer_lng",
        "seller_lat",
        "seller_lng"
    ]
).copy()

In [724]:
valid_distance.shape

(112046, 7)

In [725]:
lat1 = radians(valid_distance["customer_lat"])
lon1 = radians(valid_distance["customer_lng"])

lat2 = radians(valid_distance["seller_lat"])
lon2 = radians(valid_distance["seller_lng"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    sin(dlat / 2) ** 2
    + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
)

c = 2 * arctan2(sqrt(a), sqrt(1 - a))

valid_distance["distance_km"] = 6371 * c

In [726]:
valid_distance["distance_km"].describe()

count    112046.000000
mean        597.064456
std         590.058745
min           0.000000
25%         184.240494
50%         431.685094
75%         792.476832
max        8677.911622
Name: distance_km, dtype: float64

In [727]:
average_distance_km = (
    valid_distance
    .groupby("order_id")["distance_km"]
    .mean()
    .rename("average_distance_km")
)

In [728]:
average_distance_km.describe()

count    98154.000000
mean       601.749501
std        594.989312
min          0.000000
25%        187.382906
50%        433.815833
75%        798.679571
max       8677.911622
Name: average_distance_km, dtype: float64

In [729]:
average_distance_km.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    301.504681
00018f77f2f0320c557190d7a144bdd3    585.563937
000229ec398224ef6ca0657da4fc703e    312.343511
00024acbcdf0a6daa1e931b038114c75    293.168420
00042b26cf59d7ce69dfabb4e55b4fd9    646.163463
Name: average_distance_km, dtype: float64

In [730]:
seller_features_agg = pd.concat([
    n_unique_seller_city,
    n_unique_seller_state,
    average_distance_km
], axis=1)

seller_features_agg.head()

,n_unique_seller_city,n_unique_seller_state,average_distance_km
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,1,1,301.504681
00018f77f2f0320c557190d7a144bdd3,1,1,585.563937
000229ec398224ef6ca0657da4fc703e,1,1,312.343511
00024acbcdf0a6daa1e931b038114c75,1,1,293.168420
00042b26cf59d7ce69dfabb4e55b4fd9,1,1,646.163463


In [731]:
final_df.dtypes

order_id                                  object
customer_id                               object
order_status                              object
order_purchase_timestamp          datetime64[ns]
order_approved_at                 datetime64[ns]
order_delivered_carrier_date      datetime64[ns]
order_delivered_customer_date     datetime64[ns]
order_estimated_delivery_date     datetime64[ns]
item_count                               float64
total_price                              float64
total_freight_value                      float64
n_unique_products                        float64
n_unique_sellers                         float64
total_product_weight_g                   float64
average_product_length_cm                float64
average_product_height_cm                float64
average_product_width_cm                 float64
n_unique_categories                      float64
n_unique_seller_city                     float64
n_unique_seller_state                    float64
total_payment_value 

In [732]:
final_df = final_df.merge(
    average_distance_km,
    on="order_id",
    how="left"
)

In [733]:
print(final_df.shape)
print(final_df.columns.tolist())

(99418, 29)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_price', 'total_freight_value', 'n_unique_products', 'n_unique_sellers', 'total_product_weight_g', 'average_product_length_cm', 'average_product_height_cm', 'average_product_width_cm', 'n_unique_categories', 'n_unique_seller_city', 'n_unique_seller_state', 'total_payment_value', 'max_payment_installments', 'n_unique_payment_types', 'customer_city', 'customer_state', 'delivery_delay', 'delivery_delay_days', 'is_delayed', 'average_distance_km']


In [736]:
final_df.dtypes

order_id                                  object
customer_id                               object
order_status                              object
order_purchase_timestamp          datetime64[ns]
order_approved_at                 datetime64[ns]
order_delivered_carrier_date      datetime64[ns]
order_delivered_customer_date     datetime64[ns]
order_estimated_delivery_date     datetime64[ns]
item_count                               float64
total_price                              float64
total_freight_value                      float64
n_unique_products                        float64
n_unique_sellers                         float64
total_product_weight_g                   float64
average_product_length_cm                float64
average_product_height_cm                float64
average_product_width_cm                 float64
n_unique_categories                      float64
n_unique_seller_city                     float64
n_unique_seller_state                    float64
total_payment_value 

In [737]:
final_df["average_distance_km"].isna().sum()

np.int64(1264)

In [738]:
final_df.isna().sum()[final_df.isna().sum() > 0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
item_count                        775
total_price                       775
total_freight_value               775
n_unique_products                 775
n_unique_sellers                  775
total_product_weight_g            775
average_product_length_cm         791
average_product_height_cm         791
average_product_width_cm          791
n_unique_categories               775
n_unique_seller_city              775
n_unique_seller_state             775
total_payment_value                 1
max_payment_installments            1
n_unique_payment_types              1
delivery_delay                   2965
delivery_delay_days              2965
is_delayed                       2965
average_distance_km              1264
dtype: int64

In [739]:
final_df.to_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance).parquet",
    index=False
)

In [741]:
check_data = pd.read_parquet('/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/olist_clean.parquet')

In [742]:
check_data.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
item_count                        775
total_price                       775
total_freight_value               775
n_unique_products                 775
n_unique_sellers                  775
total_product_weight_g            775
average_product_length_cm         791
average_product_height_cm         791
average_product_width_cm          791
n_unique_categories               775
n_unique_seller_city              775
n_unique_seller_state             775
total_payment_value                 1
max_payment_installments            1
n_unique_payment_types              1
customer_city                       0
customer_state                      0
delivery_delay                   2965
delivery_del

In [744]:
current_missing = final_df.isna().sum()

current_missing[current_missing > 0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
item_count                        775
total_price                       775
total_freight_value               775
n_unique_products                 775
n_unique_sellers                  775
total_product_weight_g            775
average_product_length_cm         791
average_product_height_cm         791
average_product_width_cm          791
n_unique_categories               775
n_unique_seller_city              775
n_unique_seller_state             775
total_payment_value                 1
max_payment_installments            1
n_unique_payment_types              1
delivery_delay                   2965
delivery_delay_days              2965
is_delayed                       2965
average_distance_km              1264
dtype: int64

In [745]:
loaded_df = pd.read_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance).parquet"
)

print(loaded_df.shape)
print(loaded_df.columns.tolist())

(99418, 29)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_price', 'total_freight_value', 'n_unique_products', 'n_unique_sellers', 'total_product_weight_g', 'average_product_length_cm', 'average_product_height_cm', 'average_product_width_cm', 'n_unique_categories', 'n_unique_seller_city', 'n_unique_seller_state', 'total_payment_value', 'max_payment_installments', 'n_unique_payment_types', 'customer_city', 'customer_state', 'delivery_delay', 'delivery_delay_days', 'is_delayed', 'average_distance_km']


In [746]:
print(final_df.shape)
print(final_df.columns.tolist())

(99418, 29)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'item_count', 'total_price', 'total_freight_value', 'n_unique_products', 'n_unique_sellers', 'total_product_weight_g', 'average_product_length_cm', 'average_product_height_cm', 'average_product_width_cm', 'n_unique_categories', 'n_unique_seller_city', 'n_unique_seller_state', 'total_payment_value', 'max_payment_installments', 'n_unique_payment_types', 'customer_city', 'customer_state', 'delivery_delay', 'delivery_delay_days', 'is_delayed', 'average_distance_km']
